In [1]:
import pandas as pd
import os

In [2]:
traffic_data_path = "verkehr.csv"
traffic_data = pd.read_csv(traffic_data_path, sep=';')

plz_data_path = "plz_pop_data.csv"
plz_data = pd.read_csv(plz_data_path, sep=';')

# apply number to plz_data transformation to traffic_data (by appending column plz to traffic_data based on "Zst" == Number
plz_locality = "/home/thore/Downloads/plz_geocoord.csv"
df_plz_loc = pd.read_csv(plz_locality, sep=",", header=0)


In [3]:

plz_data.rename({"index": "plz", "Einwohner:Innen": "inhab_plz", "Bevölkerungsdichte": "density", "Einwohner im 100km-Umkreis":"inhab_100"}, inplace=True, axis=1)
plz_data.head()

,plz,inhab_plz,density,inhab_100
0,64743,3,36.555943,5143069
1,35647,4855,108.571752,4929668
2,31195,5876,83.356428,2875037
3,31084,4892,91.921198,1685539
4,27639,17093,95.001191,864845


In [4]:
traffic_data.rename({"Zst": "station_id"}, inplace=True, axis=1)
traffic_data.set_index("station_id", inplace=True)
traffic_data.head()

,Land,Strklas,Strnum,Wotag,Fahrtzw,Stunde,KFZ_R1,KFZ_R2,Monat
station_id,,,,,,,,,
8001,8,A,5,6,s,1,201,185,1
8001,8,A,5,6,s,2,450,339,1
8001,8,A,5,6,s,3,334,387,1
8001,8,A,5,6,s,4,228,282,1
8001,8,A,5,6,s,5,157,214,1


In [5]:
df_plz_loc.head()

,plz,lat,lng
0,1067,51.057550,13.717065
1,1069,51.039135,13.737675
2,1097,51.065908,13.736152
3,1099,51.087188,13.802804
4,1108,51.144324,13.799706


In [14]:
plz_data_total = df_plz_loc.copy()
# append  plz_data to plz_data_total by matching plz_data["plz"] to df_plz_loc["plz"]
plz_data_total = plz_data_total.merge(plz_data, on="plz", how="left")
plz_data_total.rename({"plz": "plz"}, inplace=True, axis=1)
plz_data_total.set_index("plz", inplace=True)
plz_data_total.head()
# save plz_data_total to plz_data_total.csv
plz_data_total.to_csv("plz_data_total.csv", sep=";", index=True)

In [6]:
import re
from pyproj import Transformer
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize geocoder
geolocator = Nominatim(user_agent="plz_extractor")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# Read file
with open('dat2.txt', 'r', encoding='utf-8') as f:
    content = f.read()

split = content.split("addBKGPoempel(")


# frop all lines that do not start with results
split = [entry for entry in split if entry.startswith("results")]
split = [entry.split("results, \"")[1].split(",<div")[0] for entry in split]

data = []
for entry in split:
    split_entry = entry.split(",")
    long = float(split_entry[0])
    lat = float(split_entry[1])
    # continue all that have blue in the name
    if "blue" in split_entry[3]:
        continue

    # get the number in the brackets

    number = re.search(r'\((\d+)\)', split_entry[3])
    if number:
        number = int(number.group(1))
    else:
        number = None
    data.append({
        "long_": long,
        "lat_": lat,
        "station_id": number
    })
    
html_data = pd.DataFrame(data)
html_data.head()


,long_,lat_,station_id
0,572102.0,5929284.0,2218
1,569966.0,5929459.0,2217
2,364845.0,5658635.0,5675
3,358801.0,5655933.0,5035
4,353633.0,5475577.0,353


In [7]:
html_data.set_index("station_id", inplace=True)

In [8]:
55.490450, 6.996040 # 9283: 47.62934906314557, 10.820777735846123
60.97620, 5.9727620 # 1108: 53.891849595827296, 10.670326976201341

(60.9762, 5.972762)

In [9]:
from pyproj import Transformer

# Transformer from EPSG:25832 (UTM zone 32N) to EPSG:4326 (WGS84 lat/lon)
transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)

# transform all "long_" and "lat_" columns to lon and lat using the transformer
def transform_coordinates(df):
    # Create new columns for transformed coordinates
    _df = pd.DataFrame()
    _df['lon'], _df['lat'] = transformer.transform(df['long_'].values, df['lat_'].values)
    _df.index = df.index
    return _df
# Apply the transformation
html_data = transform_coordinates(html_data)
# Save the transformed data to a new CSV file

html_data.head()

,lon,lat
station_id,,
2218,10.087191,53.507553
2217,10.055028,53.509414
5675,7.071179,51.063309
5035,6.986033,51.037572
353,6.981987,49.415252


In [10]:

# find closest plz to html_data and add plz to html_data
def find_closest_plz(row, df):
    # Calculate the distance between the row and all rows in df
    distances = ((df['lng'] - row['lon'])**2 + (df['lat'] - row['lat'])**2)**0.5
    # Find the index of the closest row
    closest_index = distances.idxmin()
    # Return the plz of the closest row
    return df.loc[closest_index, 'plz']
# Apply the function to each row in html_data
html_data['plz'] = html_data.apply(find_closest_plz, axis=1, df=df_plz_loc)
html_data.head()

,lon,lat,plz
station_id,,,
2218,10.087191,53.507553,22113
2217,10.055028,53.509414,20539
5675,7.071179,51.063309,51377
5035,6.986033,51.037572,51373
353,6.981987,49.415252,66571


In [11]:
# append plz to traffic_data base on "Zst" == Number

html_data = html_data.reset_index(drop=True)
# 1. PLZ basierend auf "station_id" anhängen (station_id aus traffic_data, station_id aus html_data, both index)
traffic_data_plz = traffic_data.merge(
    html_data[['plz', 'lat', 'lon']],
    how='left',
    left_index=True,
    right_index=True
)

traffic_data_plz["station_id"] = traffic_data_plz.index

# 2. Einwohner basierend auf PLZ anhängen
traffic_data_plz_inhab = traffic_data_plz.merge(
    plz_data[['plz', 'inhab_plz', 'density', 'inhab_100']],
    how='left',
    on='plz'
)
traffic_data_plz_inhab.head()

,Land,Strklas,Strnum,Wotag,Fahrtzw,Stunde,KFZ_R1,KFZ_R2,Monat,plz,lat,lon,station_id,inhab_plz,density,inhab_100
0,10,A,6,6,s,1,128,77,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
1,10,A,6,6,s,2,220,164,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
2,10,A,6,6,s,3,204,137,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
3,10,A,6,6,s,4,124,104,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
4,10,A,6,6,s,5,79,78,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0


In [12]:

# Save the merged DataFrame to a new CSV file
output_path = "merged_traffic_data.csv"
traffic_data_plz_inhab.to_csv(output_path, sep=';', index=False)
print(f"Merged data saved to {output_path}")

Merged data saved to merged_traffic_data.csv
